# 2.1.10 — Array Semantics, Mutation, and Data Ownership

**Module:** 02 — NumPy: Advanced Numerical Computing and Array Programming  
**Chapter:** 2.1 — NumPy Architecture and Fundamentals  
**Level:** Expert / Research-Oriented Foundations  
**Prerequisites:** Advanced Python object semantics; basic linear algebra; numerical analysis fundamentals.

> **Purpose.** Develop a precise, testable conceptual and computational model of array semantics, mutation, and data ownership. This notebook separates mathematical intent, Python-level semantics, storage consequences, and execution behavior. It is a standalone section of the Chapter 2.1 architecture sequence.

## 1. Learning Objectives

After studying this notebook, you should be able to:

1. State the central abstractions and identify the level of the system at which each guarantee applies.
2. Explain the relevant mathematical or computational invariants without relying on informal analogies.
3. Construct executable tests that distinguish correct behavior from plausible but invalid assumptions.
4. Identify boundary cases, side effects, dtype concerns, and complexity implications.
5. Translate the topic into a reproducible, documented numerical research workflow.

## 2. Reproducibility and Environment

The examples use the standard `numpy` package and Python standard library. They are intended for recent NumPy 2.x installations; most examples also work on current 1.26 releases, but NumPy 2.x promotion and interface details should be interpreted with the installed version in mind. Execute cells **top to bottom**. There are no external datasets, downloads, or network dependencies.

In [ ]:
import numpy as np
import sys

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
assert tuple(map(int, np.__version__.split('.')[:2])) >= (1, 24), 'Use a recent NumPy release.'

## 3. Conceptual Foundations and Formal Semantics

### 1. Three notions of sameness

**Object identity** (`is`) asks whether two variables refer to the same ndarray. **Storage overlap** (`shares_memory`) asks whether their addressed memory can overlap. **Value equality** concerns data content. These properties are independent. Distinguishing them is essential for reasoning about mutations and functional interfaces.

### 2. Basic slicing and advanced indexing

Basic slicing commonly yields views whose elements map onto the original allocation; advanced indexing generally yields copied results. However, indexed assignment has its own semantics and must not be reasoned about as though retrieving a copy and then editing it were always equivalent.

### 3. Mutation contracts

In-place operations change a target array and potentially every alias of its storage. Immutable-style APIs can avoid accidental shared-state changes by making copies at clear boundaries, but this uses memory. Conversely, aggressively in-place operations can invalidate data retained for model auditing or debugging.

### 4. Ownership flags and provenance

`OWNDATA` means an array owns its underlying data allocation in NumPy's model; `base` can identify an object retaining underlying storage, but should not be used as a universal complete alias graph. `shares_memory` examines overlap, while `may_share_memory` can conservatively report possible overlap.

### 5. Safe design strategies

Define whether functions **borrow**, **mutate**, or **return independent data**; verify non-aliasing when it is required; copy at trust boundaries; and test side effects explicitly. A read-only flag protects writes through that particular view but is not a universal immutability guarantee for all aliases.

## 4. Architectural Model and Invariants

A useful four-layer checklist for this topic:

| Layer | Question to ask | Representative invariant |
|---|---|---|
| Logical | What operation is mathematically intended? | Axes and result shape correspond to the model |
| Representation | What dtype and storage model represent the inputs? | No unsupported casting or unexpected aliasing |
| Execution | Which category of operation processes the arrays? | Results are stable under a reference implementation |
| Validation | What independent evidence establishes correctness? | Assertions, boundary tests, and numerical tolerances |

**Important:** Shape agreement and code execution are not sufficient evidence of mathematical validity. Conversely, a mathematically valid expression can be operationally impractical or numerically unstable.

## 5. Executable Investigations

Each investigation is independent in purpose; execute in sequence so shared imports are available.

### 5.1 — Basic slice aliases, fancy indexing copies

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
import numpy as np
a = np.arange(10)
basic = a[2:6]
fancy = a[[2,3,4,5]]
assert np.shares_memory(a, basic)
assert not np.shares_memory(a, fancy)
basic[0] = 100
assert a[2] == 100 and fancy[0] == 2

### 5.2 — Non-mutating function contract

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
def clipped_copy(x, low, high):
    a = np.asarray(x)
    out = np.array(a, copy=True)
    np.clip(out, low, high, out=out)
    return out
original = np.array([-4., 1., 9.])
result = clipped_copy(original, 0., 5.)
np.testing.assert_array_equal(original, [-4., 1., 9.])
np.testing.assert_array_equal(result, [0., 1., 5.])
assert not np.shares_memory(original, result)

### 5.3 — Aliasing can survive object distinction

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
a = np.arange(6.)
b = a.reshape(2, 3)
assert a is not b
assert np.shares_memory(a, b)
b[1, 1] = -99
assert a[4] == -99
print('owner flags:', a.flags.owndata, b.flags.owndata)

## 6. Edge Cases, Failure Modes, and Misconceptions

- **Logical validity is not physical contiguity.** Shape alone cannot determine whether an operation needs a copy.
- **Value equality is not identity.** Two arrays may compare equal while having independent storage, or share storage while being distinct Python objects.
- **A vectorized expression does not certify correctness.** Broadcasting and axis conventions can silently encode unintended mathematics.
- **A dtype is not a real-number model.** Overflow, finite precision, and representation-dependent transformations require explicit attention.
- **A benchmark does not imply a theorem.** Hardware, build configuration, allocation strategy, and input scale can affect conclusions.
- **Array semantics do not encode domain assumptions.** For finance or statistics, look-ahead restrictions, missing-data rules, weights, and alignment remain external responsibilities.

When evaluating an unexpected result, first inspect `shape`, `dtype`, `strides`, and storage overlap, then isolate the mathematical operation on a hand-checkable example.

## 7. Performance and Complexity Considerations

The performance question should be asked in three parts: **work complexity**, **data movement**, and **peak live memory**. A contiguous numeric array can support efficient native processing, but a particular operation may be limited by memory bandwidth, temporary allocation, scalar dispatch, or native-library parallelism. Use measurements only after verifying correctness, and never assume that a microbenchmark generalizes to different dtypes or problem sizes.

A practical benchmark should (i) isolate the operation, (ii) distinguish input setup from computation, (iii) warm up the relevant execution path, (iv) collect repeated measurements, and (v) record platform and NumPy provenance.

## 8. Analytical Interpretation

Aliasing is a correctness concern, not just a performance concern. In financial research, silently mutating the raw-return array while creating features can contaminate subsequent simulations and invalidate experiment reproducibility.

## 9. Advanced Exercises

These are open-ended expert exercises. Provide an analytical justification, a minimal counterexample where useful, and an executable validation test.

**Exercise 1.** Specify a function API contract that permits borrowing but forbids input mutation.

**Exercise 2.** Demonstrate a difference between advanced-index retrieval and repeated-index assignment.

**Exercise 3.** Explain why `writeable=False` on one alias need not freeze an entire shared allocation.

## 10. Expected Results and Validation Strategy

1. Execute every example in order; all assertions should pass on an appropriately configured NumPy installation.
2. Compare printed shapes, types, and DTypes with the surrounding mathematical and architectural claims.
3. Where a local benchmark is included, record measured values but **do not** expect a universal numeric timing.
4. Modify input shapes, dtypes, and views to construct at least one failing case; document why the original contract no longer holds.
5. Implement an independent reference for one operation and compare using an error metric appropriate to its numerical domain.

**Acceptance criteria:** the student can explain the behavior without relying on a copied output, design a counterexample to an invalid inference, and describe relevant representation and performance limitations.

## Further Reading and Verification Sources

The following are authoritative public references. API details may vary across NumPy versions; verify against the installed version when reproducing an experiment.

- [NumPy: The N-dimensional array](https://numpy.org/doc/stable/reference/arrays.ndarray.html)
- [NumPy: ndarray reference](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html)
- [NumPy: Data types](https://numpy.org/doc/stable/reference/arrays.dtypes.html)
- [NumPy: Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [NumPy: Copies and views](https://numpy.org/doc/stable/user/basics.copies.html)
- [NumPy: Universal functions](https://numpy.org/doc/stable/reference/ufuncs.html)
- [NumPy: Thread safety](https://numpy.org/doc/stable/reference/thread_safety.html)
- [NumPy: C API](https://numpy.org/doc/stable/reference/c-api/index.html)

**Scope note:** This is Chapter 2.1 (architectural foundations). Detailed DType implementation, advanced indexing, stride tricks, microarchitecture, and native C extension development are treated in Chapters 2.3, 2.5, 2.7, 2.21 and 2.23 respectively.